# OncoSense AI: Breast Cancer Risk Predictor
### *Early Detection. Smarter Decisions.*

Welcome to **OncoSense AI**, a machine learning system designed to predict whether a breast tumor is **Benign** or **Malignant** using the Wisconsin Breast Cancer Diagnostic (WBCD) dataset. This notebook is designed as an educational, end-to-end Machine Learning pipeline suitable for students, healthcare researchers, and machine learning engineers.

### Notebook Contents
1. **Dataset Inspection**: Shape, data types, missing values, duplicates, and class balance analysis.
2. **Data Preprocessing**: Data cleaning, train-test splitting (80/20), standard scaling, and label encoding.
3. **Exploratory Data Analysis (EDA)**: Professional visualizations and educational insights.
4. **Model Training**: Logistic Regression, Decision Tree, and Random Forest.
5. **Model Evaluation**: Metrics comparison (Accuracy, Precision, Recall, F1, ROC-AUC) and visualization (ROC, Confusion Matrix).
6. **Overfitting Analysis**: Training vs. Testing comparison and educational breakdown.
7. **Feature Importance**: Identifying key indicators of tumor malignancy.
8. **Interactive Risk Prediction Engine**: Patient risk assessment CLI with a colored Risk Meter and feature contribution explanation.

## Setup & Libraries
We begin by importing the necessary libraries for data processing, model training, evaluation, and plotting. We also set up Seaborn visualization styling.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

# Configure plotting aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.size': 10,
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.titlesize': 16
})

# Color Palette for visuals
PALETTE = {
    'Malignant': '#e74c3c',
    'Benign': '#2ecc71',
    'Primary': '#2c3e50',
    'Secondary': '#34495e',
    'Accent': '#f1c40f'
}
print("✔ Libraries imported and theme set successfully.")

## 1. Dataset Inspection

Before training models, it is essential to understand the shape of our data, the variable types, and check for any missing values or duplicates. 

- **Dataset shape**: Tells us how many samples (patients) and features (measurements) we have.
- **Data types**: Confirms whether our inputs are numerical or categorical.
- **Missing values**: Incomplete data can bias models; we must identify and handle them.
- **Duplicate check**: Duplicate records add redundant noise and can skew our test evaluation.
- **Class balance**: An imbalanced dataset (e.g., 99% Benign, 1% Malignant) requires special care to ensure the model doesn't just memorize the majority class.

In [ ]:
# Load the dataset
file_path = "Breast_Cancer.csv"
df = pd.read_csv(file_path)

print("=== DATASET INSPECTION ===")
print(f"✔ Shape: {df.shape[0]} rows (patients), {df.shape[1]} columns (features).")

# Data types summary
print("\n[Data Types Summary]")
print(df.dtypes.value_counts())

# Missing Value Analysis
missing_count = df.isnull().sum().sum()
print(f"\n[Missing Values]: {missing_count} missing values found.")

# Duplicate Check
duplicates = df.duplicated().sum()
print(f"[Duplicate Records]: {duplicates} duplicate records found.")

# Class Balance
class_counts = df['diagnosis'].value_counts()
print("\n[Class Balance Summary]")
for label, count in class_counts.items():
    percentage = (count / len(df)) * 100
    name = "Malignant" if label == 'M' else "Benign"
    print(f"  - {name} ({label}): {count} samples ({percentage:.2f}%)")

## 2. Data Preprocessing

We must transform the raw dataset into a clean format that machine learning algorithms can ingest:

1. **Drop Identifiers (`id`)**: The ID column contains unique sequence numbers. A model could accidentally learn that 'higher IDs mean malignancy' (spurious correlation), leading to overfitting. We drop it.
2. **Handle Missing Values**: Although this dataset is complete, we include an automated imputer using the median to handle any unexpected null entries.
3. **Remove Duplicates**: Duplicates can cause leakage between the train and test splits, exaggerating model performance. We drop them.
4. **Label Encode Target (`diagnosis`)**: Machine learning models require numerical labels. We encode Malignant ($M$) to $1$ and Benign ($B$) to $0$.
5. **Train-Test Split**: We split the dataset into $80\%$ for training and $20\%$ for testing. We use *stratified* splitting to maintain the same class ratio in both subsets.
6. **Feature Scaling (`StandardScaler`)**: Numerical features have vast differences in scale (e.g., `area_mean` is up to 2500, while `smoothness_mean` is ~0.1). Scaling transforms them to have a mean of $0$ and variance of $1$, which is critical for distance-based and gradient-descent algorithms (like Logistic Regression):
   $$z = \frac{x - \mu}{\sigma}$$

In [ ]:
data = df.copy()

# 1. Drop id and empty columns
if 'id' in data.columns:
    data = data.drop(columns=['id'])
empty_cols = [col for col in data.columns if 'Unnamed' in col]
if empty_cols:
    data = data.drop(columns=empty_cols)

# 2. Handle missing values (Median Imputation)
null_cols = data.columns[data.isnull().any()].tolist()
for col in null_cols:
    data[col] = data[col].fillna(data[col].median())

# 3. Remove duplicates
data = data.drop_duplicates()

# 4. Label Encode diagnosis
le = LabelEncoder()
data['diagnosis'] = le.fit_transform(data['diagnosis'])

# Separate features and target
X = data.drop(columns=['diagnosis'])
y = data['diagnosis']
feature_names = X.columns.tolist()

# 5. Train-Test Split (80/20 Stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 6. Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("=== PREPROCESSING COMPLETE ===")
print(f"✔ Training Set: {X_train_scaled.shape[0]} samples")
print(f"✔ Testing Set:  {X_test_scaled.shape[0]} samples")

## 3. Exploratory Data Analysis (EDA)

Visualizations help us uncover relationships between tumor characteristics and the diagnosis. We generate key visualizations below.

In [ ]:
# Setup colors for plotting
colors = [PALETTE['Benign'], PALETTE['Malignant']]

# 3.1 Diagnosis Distribution
plt.figure(figsize=(6, 4))
ax = sns.countplot(x='diagnosis', hue='diagnosis', data=df, palette=colors, legend=False)
plt.title('Tumor Diagnosis Distribution (Class Balance)', pad=15)
plt.xlabel('Diagnosis (B = Benign, M = Malignant)')
plt.ylabel('Count')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height() + 5),
                ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontweight='bold')
plt.tight_layout()
plt.show()
print("💡 INSIGHT 1: The dataset has a mild class imbalance (~63% Benign, ~37% Malignant).")
print("💡 INSIGHT 2: Models should be evaluated on F1-Score and Recall, not just raw accuracy.")

In [ ]:
# 3.2 Correlation Heatmap (Top 12 features for readability)
plt.figure(figsize=(10, 8))
numeric_df = df.copy()
if 'id' in numeric_df.columns:
    numeric_df = numeric_df.drop(columns=['id'])
if not pd.api.types.is_numeric_dtype(numeric_df['diagnosis']):
    numeric_df['diagnosis'] = LabelEncoder().fit_transform(numeric_df['diagnosis'].astype(str))

correlations = numeric_df.corr()
top_corr_features = correlations['diagnosis'].abs().sort_values(ascending=False).index[:12]
sns.heatmap(numeric_df[top_corr_features].corr(), annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Correlation Heatmap (Top 12 Most Correlated Features)', pad=15)
plt.tight_layout()
plt.show()
print("💡 INSIGHT 1: Concave points, perimeter, area, and radius have extremely high correlation with diagnosis (>0.70).")
print("💡 INSIGHT 2: High multicollinearity exists (e.g., radius, perimeter, and area are highly correlated with each other).")

In [ ]:
# 3.3 Feature Distribution (Radius Mean & Texture Mean)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.kdeplot(data=df, x='radius_mean', hue='diagnosis', fill=True, palette=colors, alpha=0.5, ax=axes[0], common_norm=False)
axes[0].set_title('Distribution of Radius Mean by Diagnosis')
axes[0].set_xlabel('Radius Mean (mm)')

sns.kdeplot(data=df, x='texture_mean', hue='diagnosis', fill=True, palette=colors, alpha=0.5, ax=axes[1], common_norm=False)
axes[1].set_title('Distribution of Texture Mean by Diagnosis')
axes[1].set_xlabel('Texture Mean (Gray-scale value)')
plt.tight_layout()
plt.show()
print("💡 INSIGHT 1: Malignant tumors typically have a significantly larger average radius (>15mm) than benign tumors.")
print("💡 INSIGHT 2: Texture mean (roughness) overlaps heavily, indicating it is a weaker predictor on its own.")

In [ ]:
# 3.4 Pair Plot (Top 4 Important Features)
top_features = correlations['diagnosis'].abs().sort_values(ascending=False).index[1:5].tolist()
pair_grid = sns.pairplot(df, vars=top_features, hue='diagnosis', palette=colors, diag_kind='kde', height=2.5)
pair_grid.fig.suptitle('Pair Plot of Top 4 Distinctive Features', y=1.02, fontsize=14)
plt.show()
print("💡 INSIGHT 1: Malignant classes separate well from benign classes along concave points and perimeter features.")
print("💡 INSIGHT 2: Linear boundaries can separate a large portion of the samples, making Logistic Regression a strong baseline.")

In [ ]:
# 3.5 Outlier Detection (Boxplots)
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
key_features = ['radius_mean', 'area_mean', 'perimeter_mean', 'concavity_mean']
for i, feature in enumerate(key_features):
    row, col = i // 2, i % 2
    sns.boxplot(x='diagnosis', y=feature, hue='diagnosis', data=df, palette=colors, ax=axes[row, col], legend=False)
    axes[row, col].set_title(f'Outlier Detection - {feature}')
    axes[row, col].set_xlabel('Diagnosis')
    axes[row, col].set_ylabel(feature)
plt.tight_layout()
plt.show()
print("💡 INSIGHT 1: Benign tumors have several outliers on the higher end of size features, representing unusually large benign cases.")
print("💡 INSIGHT 2: Robust modeling algorithms or scaling are necessary to prevent outliers from pulling the decision boundary.")

## 4. Model Training & Evaluation

We train three diverse classification models:
1. **Logistic Regression**: A linear classification algorithm modeled using the sigmoid function:
   $$P(y=1) = \frac{1}{1 + e^{-(\mathbf{w}^T\mathbf{x} + b)}}$$
2. **Decision Tree**: A non-linear classifier that splits data on features using **Gini Impurity** to maximize information gain:
   $$Gini = 1 - \sum (p_i)^2$$
3. **Random Forest**: An ensemble of $100$ Decision Trees trained on bootstrapped subsets of the data (Bagging) to reduce variance and control overfitting.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=10000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

results = {}
roc_curves = {}
confusion_matrices = {}

for name, model in models.items():
    # Train
    model.fit(X_train_scaled, y_train)
    
    # Predict
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    
    # Training accuracy for overfitting check
    y_train_pred = model.predict(X_train_scaled)
    train_acc = accuracy_score(y_train, y_train_pred)

    # Test metrics
    test_acc = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    results[name] = {
        'Train Accuracy': train_acc,
        'Test Accuracy': test_acc,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1,
        'ROC-AUC Score': auc,
        'model_object': model
    }
    
    # Store ROC & CM
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_curves[name] = (fpr, tpr, auc)
    confusion_matrices[name] = confusion_matrix(y_test, y_pred)

    print(f"=== {name} Classification Report ===")
    print(classification_report(y_test, y_pred, target_names=['Benign', 'Malignant']))
    print("-" * 50)

### Model Comparison Summary

In [ ]:
comparison_df = pd.DataFrame(results).T.drop(columns=['model_object'])
display(comparison_df)

# Plot ROC Curves
plt.figure(figsize=(8, 6))
for name, (fpr, tpr, auc) in roc_curves.items():
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})', lw=2)
plt.plot([0, 1], [0, 1], 'k--', label='Random Guessing (AUC = 0.500)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Recall / Sensitivity)')
plt.title('ROC Curves Comparison')
plt.legend(loc="lower right")
plt.show()

# Plot Confusion Matrices Side-by-Side
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, (name, cm) in enumerate(confusion_matrices.items()):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i], cbar=False,
                xticklabels=['Benign', 'Malignant'], yticklabels=['Benign', 'Malignant'])
    axes[i].set_title(f'{name} Confusion Matrix')
    axes[i].set_xlabel('Predicted Label')
    axes[i].set_ylabel('True Label')
plt.tight_layout()
plt.show()

best_model_name = max(results, key=lambda k: results[k]['F1 Score'])
best_model_obj = results[best_model_name]['model_object']
print(f"👑 SELECTED MODEL: {best_model_name} is automatically selected as the best predictor based on F1 Score.")

## 5. Overfitting Analysis

Overfitting occurs when a model memorizes details and noise of the training data to the extent that it fails to perform on new testing cases. 

We check for overfitting by examining the **Generalization Gap**:
$$\text{Generalization Gap} = \text{Train Accuracy} - \text{Test Accuracy}$$

In [ ]:
print('=== OVERFITTING ANALYSIS ===\n')
for name, metrics in results.items():
    train_acc = metrics['Train Accuracy'] * 100
    test_acc = metrics['Test Accuracy'] * 100
    gap = train_acc - test_acc
    print(f'👉 {name}:')
    print(f'   - Training Accuracy: {train_acc:.2f}%')
    print(f'   - Testing Accuracy:  {test_acc:.2f}%')
    print(f'   - Generalization Gap: {gap:.2f}%')
    if gap > 5.0:
        print('   🚨 STATUS: POTENTIAL OVERFITTING. The model is memorizing the training set.')
    else:
        print('   ✅ STATUS: STABLE GENERALIZATION. Generalization gap is healthy.')
    print()

## 6. Feature Importance

Understanding *why* the model makes a prediction is crucial in medical AI. We extract and plot feature importance scores for our selected best model.

In [ ]:
plt.figure(figsize=(10, 6))
if hasattr(best_model_obj, 'feature_importances_'):
    importances = best_model_obj.feature_importances_
    title = f'Feature Importances (Gini Impurity) - {best_model_name}'
    label = 'Importance Score'
elif hasattr(best_model_obj, 'coef_'):
    importances = np.abs(best_model_obj.coef_[0])
    title = f'Feature Importances (Coefficients) - {best_model_name}'
    label = 'Absolute Coefficient Magnitude'
else:
    importances = np.ones(len(feature_names))
    title = 'Equal Importances'
    label = 'Score'

feat_series = pd.Series(importances, index=feature_names)
top_feats = feat_series.sort_values(ascending=True).tail(12)
top_feats.plot(kind='barh', color=plt.cm.coolwarm(np.linspace(0.8, 0.2, len(top_feats))))
plt.title(title, pad=15)
plt.xlabel(label)
plt.ylabel('Features')
plt.tight_layout()
plt.show()
print("💡 TOP INDICATORS OF MALIGNANCY:")
for f_name, f_val in top_feats.sort_values(ascending=False).items():
    print(f"  - {f_name}: Score {f_val:.4f}")

## 7. Interactive Risk Prediction Engine

Run the cell below to launch the **OncoSense AI Risk Engine**. It allows you to select test cases from the dataset, see a visual colored risk meter, and get a list of features contributing to the risk.

In [ ]:
def get_risk_meter_str(probability):
    segments = int(round(probability * 10))
    meter = "■" * segments + "□" * (10 - segments)
    if probability < 0.35:
        risk_level = "🟢 LOW RISK"
    elif probability < 0.70:
        risk_level = "🟡 MODERATE RISK"
    else:
        risk_level = "🔴 HIGH RISK"
    return f"[{meter}] {probability*100:.1f}% - {risk_level}"

def explain_local_prediction(patient_scaled, benign_mean_scaled):
    if hasattr(best_model_obj, 'feature_importances_'):
        importances = best_model_obj.feature_importances_
    elif hasattr(best_model_obj, 'coef_'):
        importances = np.abs(best_model_obj.coef_[0])
        importances = importances / np.sum(importances)
    else:
        importances = np.ones(len(feature_names)) / len(feature_names)
        
    deviations = patient_scaled - benign_mean_scaled
    contributions = importances * deviations
    
    contrib_df = pd.DataFrame({
        'Feature': feature_names,
        'Patient Value': patient_scaled,
        'Benign Mean': benign_mean_scaled,
        'Contribution': contributions
    })
    top_risk = contrib_df.sort_values(by='Contribution', ascending=False).head(3)
    return [f"• {row['Feature'].replace('_', ' ').title()}" for _, row in top_risk.iterrows()]

def display_patient_report(idx):
    patient_scaled = X_test_scaled[idx].reshape(1, -1)
    prob = best_model_obj.predict_proba(patient_scaled)[0, 1]
    pred = best_model_obj.predict(patient_scaled)[0]
    pred_label = "Malignant" if pred == 1 else "Benign"
    true_label = "Malignant" if y_test.iloc[idx] == 1 else "Benign"
    confidence = prob if pred == 1 else (1 - prob)
    benign_mean_scaled = X_test_scaled[y_test.reset_index(drop=True) == 0].mean(axis=0)
    
    print("-" * 50)
    print(f"          ONCOSENSE PATIENT REPORT: INDEX #{idx}          ")
    print("-" * 50)
    print(f"Actual Clinical Diagnosis: {true_label}")
    print(f"Prediction:               {pred_label}")
    print(f"Confidence:               {confidence*100:.2f}%")
    print(f"AI Risk Meter:            {get_risk_meter_str(prob)}")
    print("\nTop Contributing Risk Factors:")
    for c in explain_local_prediction(X_test_scaled[idx], benign_mean_scaled):
        print(c)
    print("-" * 50)

# Show predictions for a sample Malignant case and a sample Benign case in the test set
y_test_reset = y_test.reset_index(drop=True)
m_idx = y_test_reset[y_test_reset == 1].index[0]
b_idx = y_test_reset[y_test_reset == 0].index[0]

print("\n=== DEMONSTRATING TYPICAL MALIGNANT PATIENT EVALUATION ===")
display_patient_report(m_idx)

print("\n=== DEMONSTRATING TYPICAL BENIGN PATIENT EVALUATION ===")
display_patient_report(b_idx)